# The Perceptron Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: The Perceptron class

In [ ]:
```python

class Perceptron:

    def __init__(self, n_inputs, learning_rate=0.1):

        self.weights = [0.0] * n_inputs

        self.bias = 0.0

        self.lr = learning_rate

    def predict(self, inputs):

        total = sum(w * x for w, x in zip(self.weights, inputs))

        total += self.bias

        return 1 if total >= 0 else 0

    def train(self, training_data, epochs=100):

        for epoch in range(epochs):

            errors = 0

            for inputs, target in training_data:

                prediction = self.predict(inputs)

                error = target - prediction

                if error != 0:

                    errors += 1

                    for i in range(len(self.weights)):

                        self.weights[i] += self.lr * error * inputs[i]

                    self.bias += self.lr * error

            if errors == 0:

                print(f"Converged at epoch {epoch + 1}")

                return

        print(f"Did not converge after {epochs} epochs")

In [ ]:
```

### Step 2: Train on logic gates

In [ ]:
```python

and_data = [

    ([0, 0], 0),

    ([0, 1], 0),

    ([1, 0], 0),

    ([1, 1], 1),

]

or_data = [

    ([0, 0], 0),

    ([0, 1], 1),

    ([1, 0], 1),

    ([1, 1], 1),

]

not_data = [

    ([0], 1),

    ([1], 0),

]

print("=== AND Gate ===")

p_and = Perceptron(2)

p_and.train(and_data)

for inputs, _ in and_data:

    print(f"  {inputs} -> {p_and.predict(inputs)}")

print("\n=== OR Gate ===")

p_or = Perceptron(2)

p_or.train(or_data)

for inputs, _ in or_data:

    print(f"  {inputs} -> {p_or.predict(inputs)}")

print("\n=== NOT Gate ===")

p_not = Perceptron(1)

p_not.train(not_data)

for inputs, _ in not_data:

    print(f"  {inputs} -> {p_not.predict(inputs)}")

In [ ]:
```

### Step 3: Watch XOR fail

In [ ]:
```python

xor_data = [

    ([0, 0], 0),

    ([0, 1], 1),

    ([1, 0], 1),

    ([1, 1], 0),

]

print("\n=== XOR Gate (single perceptron) ===")

p_xor = Perceptron(2)

p_xor.train(xor_data, epochs=1000)

for inputs, expected in xor_data:

    result = p_xor.predict(inputs)

    status = "OK" if result == expected else "WRONG"

    print(f"  {inputs} -> {result} (expected {expected}) {status}")

In [ ]:
```

It will never converge. This is the hard proof that a single perceptron cannot learn XOR.

### Step 4: Solve XOR with two layers

The trick: XOR = (x1 OR x2) AND NOT (x1 AND x2). Combine three perceptrons:

In [ ]:
```mermaid

graph LR

    x1["x1"] --> OR["OR neuron"]

    x1 --> NAND["NAND neuron"]

    x2["x2"] --> OR

    x2 --> NAND

    OR --> AND["AND neuron"]

    NAND --> AND

    AND --> out["output"]

In [ ]:
```

In [ ]:
```python

def xor_network(x1, x2):

    or_neuron = Perceptron(2)

    or_neuron.weights = [1.0, 1.0]

    or_neuron.bias = -0.5

    nand_neuron = Perceptron(2)

    nand_neuron.weights = [-1.0, -1.0]

    nand_neuron.bias = 1.5

    and_neuron = Perceptron(2)

    and_neuron.weights = [1.0, 1.0]

    and_neuron.bias = -1.5

    hidden1 = or_neuron.predict([x1, x2])

    hidden2 = nand_neuron.predict([x1, x2])

    output = and_neuron.predict([hidden1, hidden2])

    return output

print("\n=== XOR Gate (multi-layer network) ===")

for inputs, expected in xor_data:

    result = xor_network(inputs[0], inputs[1])

    print(f"  {inputs} -> {result} (expected {expected})")

In [ ]:
```

All four cases correct. Stacking perceptrons into layers creates decision boundaries that no single perceptron can produce.

### Step 5: Train a Two-Layer Network

Step 4 hand-wired the weights. That works for XOR, but not for real problems where you don't know the right weights in advance. The fix: replace the step function with sigmoid and learn the weights automatically through backpropagation.

In [ ]:
```python

class TwoLayerNetwork:

    def __init__(self, learning_rate=0.5):

        import random

        random.seed(0)

        self.w_hidden = [[random.uniform(-1, 1), random.uniform(-1, 1)] for _ in range(2)]

        self.b_hidden = [random.uniform(-1, 1), random.uniform(-1, 1)]

        self.w_output = [random.uniform(-1, 1), random.uniform(-1, 1)]

        self.b_output = random.uniform(-1, 1)

        self.lr = learning_rate

    def sigmoid(self, x):

        import math

        x = max(-500, min(500, x))

        return 1.0 / (1.0 + math.exp(-x))

    def forward(self, inputs):

        self.inputs = inputs

        self.hidden_outputs = []

        for i in range(2):

            z = sum(w * x for w, x in zip(self.w_hidden[i], inputs)) + self.b_hidden[i]

            self.hidden_outputs.append(self.sigmoid(z))

        z_out = sum(w * h for w, h in zip(self.w_output, self.hidden_outputs)) + self.b_output

        self.output = self.sigmoid(z_out)

        return self.output

    def train(self, training_data, epochs=10000):

        for epoch in range(epochs):

            total_error = 0

            for inputs, target in training_data:

                output = self.forward(inputs)

                error = target - output

                total_error += error ** 2

                d_output = error * output * (1 - output)

                saved_w_output = self.w_output[:]

                hidden_deltas = []

                for i in range(2):

                    h = self.hidden_outputs[i]

                    hd = d_output * saved_w_output[i] * h * (1 - h)

                    hidden_deltas.append(hd)

                for i in range(2):

                    self.w_output[i] += self.lr * d_output * self.hidden_outputs[i]

                self.b_output += self.lr * d_output

                for i in range(2):

                    for j in range(len(inputs)):

                        self.w_hidden[i][j] += self.lr * hidden_deltas[i] * inputs[j]

                    self.b_hidden[i] += self.lr * hidden_deltas[i]

In [ ]:
```

In [ ]:
```python

net = TwoLayerNetwork(learning_rate=2.0)

net.train(xor_data, epochs=10000)

for inputs, expected in xor_data:

    result = net.forward(inputs)

    predicted = 1 if result >= 0.5 else 0

    print(f"  {inputs} -> {result:.4f} (rounded: {predicted}, expected {expected})")

In [ ]:
```

Two key differences from Step 4. First, sigmoid replaces the step function -- it's smooth, so gradients exist. Second, the `train` method propagates error backward from output to hidden layer, adjusting every weight proportionally to its contribution to the error. That's backpropagation in 20 lines.

This is the bridge to Lesson 03. The math behind `d_output` and `hidden_deltas` is the chain rule applied to the network graph. We'll derive it properly there.

## Exercises

In [ ]:
1. Train a perceptron on a NAND gate (the universal gate - any logic circuit can be built from NAND). Verify its weights and bias form a valid decision boundary.
2. Modify the Perceptron class to track the decision boundary (w1*x1 + w2*x2 + b = 0) at each epoch. Print how the line shifts during training on the AND gate.
3. Build a 3-input perceptron that outputs 1 only when at least 2 of the 3 inputs are 1 (a majority vote function). Is this linearly separable? Why?